In [ ]:
# Activation function definitions
from typing import Callable

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

In [ ]:
font_manager.fontManager.addfont("bp-typst/res/LibertinusSerif-Regular.ttf")
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Libertinus Serif"]
plt.rcParams["font.size"] = 14


def plot_graph(
    ax,
    f: Callable[[np.ndarray], np.ndarray],
    color: str,
    title: str,
    x_range: tuple[float, float],
    y_range: tuple[float, float],
) -> None:
    x = np.linspace(x_range[0], x_range[1], 200)
    ax.plot(x, f(x), color=color, linewidth=3)
    ax.axvline(0, color="gray", linewidth=0.5)
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylim(y_range[0], y_range[1])
    ax.grid(True)


def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-x))


def relu(x: np.ndarray) -> np.ndarray:
    return np.maximum(0, x)


def binary_step(x: np.ndarray) -> np.ndarray:
    return np.where(x >= 0, 1, 0)


def identity(x: np.ndarray) -> np.ndarray:
    return x


fig, axes = plt.subplots(2, 2, figsize=(8, 8))
plot_graph(axes[0][0], identity, "C0", "Identity", (-1.0, 1.0), (-1.1, 1.1))
plot_graph(axes[0][1], sigmoid, "C1", "Sigmoid", (-6, 6), (-0.1, 1.1))
plot_graph(axes[1][0], binary_step, "C2", "Binary Step", (-2, 2), (-0.1, 1.1))
plot_graph(axes[1][1], relu, "C3", "ReLU", (-1.0, 1.0), (-0.1, 1.1))


axes[0][0].set_ylabel("activation(x)")
axes[1][0].set_ylabel("activation(x)")
plt.tight_layout(rect=(0, 0.03, 1, 0.95))
plt.savefig("bp-typst/fig/graphs/activations.svg")
plt.show()

In [ ]:
# Simple regression example: house size -> house price
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

# Synthetic training data (size in m^2, price in thousand EUR)
x_size = np.linspace(35, 200, 28)
true_price = 1.9 * x_size + 40
y_price = true_price + rng.normal(0, 22, size=x_size.shape)

# Fit a linear regression line y = a*x + b
a, b = np.polyfit(x_size, y_price, 1)
x_line = np.linspace(x_size.min(), x_size.max(), 200)
y_line = a * x_line + b

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_size, y_price, s=45, alpha=0.8, label="Observed houses")
ax.plot(x_line, y_line, color="C3", linewidth=2.5, label=f"Regression line: y = {a:.2f}x + {b:.2f}")

ax.set_xlabel("House size (m^2)")
ax.set_ylabel("Price (thousand EUR)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig("bp-typst/fig/graphs/house-regression.svg")
plt.show()

In [ ]:
# Simple classification example with a linear decision boundary
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(17)

# Two synthetic classes in 2D (cats vs tacos)
n = 60
mean_cat = np.array([16.8, 0.85])
mean_taco = np.array([13.4, 0.25])
cov = np.array([[0.8, 0.01], [0.01, 0.02]])
x_cat = rng.multivariate_normal(mean_cat, cov, n)
x_taco = rng.multivariate_normal(mean_taco, cov, n)

# Keep cuteness in [0, 1] for both classes
x_cat[:, 1] = np.clip(x_cat[:, 1], 0.0, 1.0)
x_taco[:, 1] = np.clip(x_taco[:, 1], 0.0, 1.0)

# Simple linear separator derived from class means
w = mean_taco - mean_cat
b = -0.5 * (mean_taco + mean_cat) @ w

# Grid for probability background
x_min = min(x_cat[:, 0].min(), x_taco[:, 0].min()) - 0.8
x_max = max(x_cat[:, 0].max(), x_taco[:, 0].max()) + 0.8
y_min = 0.0
y_max = 1.0
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
scores = w[0] * xx + w[1] * yy + b
probs = 1 / (1 + np.exp(-scores))

fig, ax = plt.subplots(figsize=(7.5, 5.5))
contour = ax.contourf(xx, yy, probs, levels=20, cmap="RdBu", alpha=0.35)
ax.contour(xx, yy, probs, levels=[0.5], colors="k", linewidths=1.5)

ax.scatter(x_cat[:, 0], x_cat[:, 1], s=35, color="C1", label="Cat")
ax.scatter(x_taco[:, 0], x_taco[:, 1], s=35, color="C0", label="Taco")

cbar = fig.colorbar(contour, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("P(Taco)")

ax.set_xlabel("Protein content (%)")
ax.set_ylabel("Cuteness (0-1)")
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.grid(True, alpha=0.25)
ax.legend()
plt.tight_layout()
plt.savefig("bp-typst/fig/graphs/classification-boundary.svg")
plt.show()

In [ ]:
# Classification probabilities as a styled bar chart
import numpy as np
import matplotlib.pyplot as plt

labels = ["Taco", "Cat", "Goat", "Cheese", "Pizza"]
percentages = np.array([80, 2, 2, 10, 6])
colors = ["#2C7FB8", "#7FCDBB", "#C7E9B4", "#FDD49E", "#FC8D59"]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
bars = ax.bar(labels, percentages, color=colors, edgecolor="#1f1f1f", linewidth=0.6)

# Add value labels above bars
for bar, value in zip(bars, percentages):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1.2,
        f"{value}%",
        ha="center",
        va="bottom",
        fontsize=11,
    )

ax.set_ylabel("Probability (%)")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("bp-typst/fig/graphs/classification-probabilities.svg")
plt.show()